# Entrenamiento del agente DQN

## Configuraciones de entorno

In [ ]:
!pip install gymnasium[atari] ale-py autorom
!AutoROM --accept-license

In [ ]:
import os
import zipfile
import gymnasium as gym
import ale_py
import shimmy
import numpy as np
from stable_baselines3 import DQN
from stable_baselines3.common.env_util import make_atari_env
from stable_baselines3.common.vec_env import VecFrameStack, SubprocVecEnv
from stable_baselines3.common.callbacks import EvalCallback, CheckpointCallback

## REGISTRO DE ENTORNOS

In [ ]:
gym.register_envs(ale_py)

## CONFIGURACIÓN

In [ ]:
ENV_ID = "BreakoutNoFrameskip-v4"
STEPS_TRAINING = 10_000_000
N_ENVS = 8 

## Carpetas organizadas

In [ ]:
BASE_DIR = "./entrenamiento_breakout_10M"
LOG_DIR = f"{BASE_DIR}/monitor_logs"
TENSORBOARD_DIR = f"{BASE_DIR}/tensorboard"
MODELS_DIR = f"{BASE_DIR}/modelos"

os.makedirs(LOG_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)


## 1. ENTORNOS

In [ ]:
# Entorno de Entrenamiento:
# 'monitor_dir' guardará un CSV con la historia de cada partida
train_env = make_atari_env(
    ENV_ID, 
    n_envs=N_ENVS, 
    seed=0, 
    vec_env_cls=SubprocVecEnv,
    monitor_dir=LOG_DIR 
)
train_env = VecFrameStack(train_env, n_stack=4)

# Entorno de Evaluación:
eval_env = make_atari_env(ENV_ID, n_envs=1, seed=42)
eval_env = VecFrameStack(eval_env, n_stack=4)

## 2. CALLBACKS DE SEGURIDAD Y ESTADÍSTICA

In [ ]:
# A. EvalCallback: Guarda el 'best_model.zip' y métricas de evaluación (.npz)
eval_callback = EvalCallback(
    eval_env,
    best_model_save_path=MODELS_DIR,
    log_path=MODELS_DIR,
    eval_freq=50_000,       # Evaluar cada 50k pasos
    n_eval_episodes=10,     # Promediar 10 partidas para ser precisos
    deterministic=True,
    render=False
)

# B. CheckpointCallback: Guarda una copia del agente cada 1M de pasos
checkpoint_callback = CheckpointCallback(
    save_freq=1_000_000,
    save_path=MODELS_DIR,
    name_prefix="dqn_bk_checkpoint"
)

## 3. MODELO

model = DQN(
    "CnnPolicy",
    train_env,
    # RAM KAGGLE: 250k es seguro. 500k arriesgado si usas GPU T4 x2. 
    # Si la RAM se llena, baja esto a 100_000.
    buffer_size=250_000, 
    
    learning_rate=1e-4,
    learning_starts=50_000,    # Llenar bien la memoria antes de aprender
    batch_size=32,
    train_freq=4,
    gradient_steps=1,
    target_update_interval=10_000, # Estabilidad a largo plazo
    exploration_fraction=0.1,      # 1M de pasos explorando (10% del total)
    exploration_final_eps=0.01,
    verbose=0,                     # Silencioso para no llenar el log de Kaggle
    device="cuda",
    tensorboard_log=TENSORBOARD_DIR # <--- Guarda TODAS las estadísticas internas
)

## 4. EJECUCIÓN

In [ ]:
print(f"INICIANDO ENTRENAMIENTO DE {STEPS_TRAINING} PASOS")

try:
    model.learn(total_timesteps=STEPS_TRAINING, callback=[eval_callback, checkpoint_callback])
    print("Entrenamiento finalizado exitosamente.")
    
    # Guardar modelo final explícito
    model.save(f"{MODELS_DIR}/dqn_breakout_FINAL_10M")

except Exception as e:
    print(f"Ocurrió un error: {e}")

finally:
    train_env.close()
    eval_env.close()


## 5. EMPAQUETADO FINAL

In [ ]:
def zip_directory(folder_path, zip_path):
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
        for root, dirs, files in os.walk(folder_path):
            for file in files:
                zipf.write(os.path.join(root, file), 
                           os.path.relpath(os.path.join(root, file), 
                           os.path.join(folder_path, '..')))

# Creamos un archivo .zip con los datos
zip_directory(BASE_DIR, "RESULTADOS_BREAKOUT_10M.zip")